# 1. Importing the Labarary

In [3]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

# 2 Data loading and Understanding

In [7]:
# load the csv data to a pandas dataframe 
df = pd.read_csv(r'C:\Users\Rohit\Downloads\WA_Fn-UseC_-Telco-Customer-Churn.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Rohit\\Downloads\\WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
df.head(2)

In [ ]:
df.info()

In [ ]:
# dropping CustomerId column as this is not required for modeling
df = df.drop(columns=["customerID"])

In [ ]:
df.head(2)

In [ ]:
df.columns

In [ ]:
# printing the unique values in all the columns
for col in df.columns:
    print(col, df[col].unique())
    print("-"*50)

In [ ]:
print(df.isnull().sum())

In [ ]:
# df["TotalCharges"] = df["TotalCharges"].astype(float)

In [ ]:
df[df["TotalCharges"]==" "]

In [ ]:
len(df[df["TotalCharges"]==" "])

In [ ]:
df["TotalCharges"] = df["TotalCharges"].replace({" ": "0.0"})

In [ ]:
df["TotalCharges"] = df["TotalCharges"].astype(float)

In [ ]:
df.info()

In [ ]:
# cheking the class Distribution of target column
print(df["Churn"].value_counts())

# Insights
# 1. Customer Id  removed as it not required for Modelling
# 2. No missing value in the dataset
# 3. missing value in the totalcharges column were replaced with 0
# 4. class imbalance identified in the taget

# 3. Exploratory Data Analysis (EDA)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
df.describe() # only one work by chatogerial columns 

# Numerical features-Analysis

In [ ]:
# understand the distribution of teh numerical features
def plot_histogram(df, column_name):
    plt.figure(figsize=(5,3))
    sns.histplot(df[column_name], kde=True)
    plt.title(f"Distribution of {column_name}")
    
    # calculate the mean and median values for the columns
    col_mean = df[column_name].mean()
    col_median = df[column_name].median()
    
    # add vertical line for mean and median
    plt.axvline(col_mean, color="red", linestyle="--", label="mean")
    plt.axvline(col_median, color="green", linestyle="-", label="median")

    plt.legend()
    plt.show()

In [ ]:
plot_histogram(df, "tenure")

In [ ]:
plot_histogram(df, "MonthlyCharges")

In [ ]:
plot_histogram(df, "TotalCharges")

# Box plot for numerical Features

In [ ]:
def plot_boxplot(df, column_name):
    plt.figure(figsize=(5,3))
    sns.boxplot(y=df[column_name])
    plt.title(f"Box plot of {column_name}")
    plt.ylabel(column_name)
    plt.show

In [ ]:
plot_boxplot(df, "tenure")

In [ ]:
plot_boxplot(df, "MonthlyCharges")

In [ ]:
plot_boxplot(df, "TotalCharges")

# Correlation heatmap for numerical columns

In [ ]:
# correlation matrix - heatmap
plt.figure(figsize=(8,4))
sns.heatmap(df[["tenure", "MonthlyCharges", "TotalCharges"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# Categorical features-Analysis

In [ ]:
df.info()

# Count for Catogorical columns

In [ ]:
object_cols = df.select_dtypes(include="object").columns.to_list()
object_cols = ["SeniorCitizen"] + object_cols

for col in object_cols:
    plt.figure(figsize=(5,3))
    sns.countplot(x=df[col])
    plt.title(f"Count plot of {col}")
    plt.show()

# 4 Data Preprocesssing

In [ ]:
df.head(3)

# Label encoding of target column

In [ ]:
df["Churn"] = df["Churn"].replace({"Yes": 1, "No": 0})

In [ ]:
df.head(3)

In [ ]:
print(df["Churn"].value_counts())

# label encoding of categorical features

In [ ]:
# idenifying columns with object data type
object_columns = df.select_dtypes(include="object").columns

In [ ]:
print(object_columns)

In [ ]:
# initilize a dictionry to save the encoders
encoders = {}

# apply label encoding and store the encoders
for column in object_columns:
    label_encoder = LabelEncoder()
    df[column] = label_encoder.fit_transform(df[column])
    encoders[column] = label_encoder
# save the encoders to a pickle file
    with open("encoders.pkl", "wb") as f:
        pickle.dump(encoders, f)

In [ ]:
encoders

In [ ]:
df.head()

# training and test data split

In [ ]:
# spliting the features and target
x = df.drop(columns=["Churn"])
y = df["Churn"]

In [ ]:
# split traing and test data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
print(y_train.shape)

In [ ]:
print(y_train.value_counts())

# Synthetic minority Oversampling TEchnique (SMOTE)

In [ ]:
smote = SMOTE(random_state=42)

In [ ]:
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

In [ ]:
print(y_train_smote.shape)

In [ ]:
print(y_train_smote.value_counts())

# 5 Model training

In [ ]:
# training with default hyperparameters

In [ ]:
# dictionry of models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42)
}

In [ ]:
# Dictionry to store validation results
cv_scores = {}

# perform 5-flod cross validation for each model
for model_name, model in models.items():
    print(f"Training {model_name} with default parameters")
    scores = cross_val_score(model, x_train_smote, y_train_smote, cv=5, scoring="accuracy")
    cv_scores[model_name] = scores
    print(f"{model_name} cross-validation accuracy: {np.mean(scores):.2f}")